In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
from src.config import Config
from src.utils.seed import set_seed
from src.utils.io_helpers import save_checkpoint, save_json
from src.dataset import get_dataloaders
from src.models.dcgan import DCGANGenerator, DCGANDiscriminator, weights_init_dcgan
from src.trainers.train_gan import train_gan
from src.evaluation.metrics import evaluate_gan_latency

set_seed(Config.SEED)
Config.ensure_dirs()
train_loader, _, _ = get_dataloaders(batch_size=Config.BATCH_SIZE, seed=Config.SEED, num_workers=0)

dcgan_gen = DCGANGenerator(
    noise_dim=Config.DCGAN_NOISE_DIM,
    feature_maps=Config.DCGAN_FEATURE_MAPS,
    img_channels=Config.IMG_CHANNELS,
)
dcgan_disc = DCGANDiscriminator(
    feature_maps=Config.DCGAN_FEATURE_MAPS,
    img_channels=Config.IMG_CHANNELS,
)

dcgan_gen.apply(weights_init_dcgan)
dcgan_disc.apply(weights_init_dcgan)

train_out = train_gan(
    generator=dcgan_gen,
    discriminator=dcgan_disc,
    train_loader=train_loader,
    epochs=Config.DCGAN_EPOCHS,
    lr=Config.DCGAN_LR,
    betas=Config.GAN_BETAS,
    noise_dim=Config.DCGAN_NOISE_DIM,
    device=Config.DEVICE,
    model_name="DCGAN",
)

latency_ms = evaluate_gan_latency(dcgan_gen, Config.DCGAN_NOISE_DIM, Config.DEVICE)
print(f"DCGAN Generator Latency (batch=64): {latency_ms} ms")

save_checkpoint(dcgan_gen, Config.CHECKPOINT_DIR / "dcgan_gen.pth")
save_checkpoint(dcgan_disc, Config.CHECKPOINT_DIR / "dcgan_disc.pth")
save_json(
    {
        "model": "Deep Convolutional GAN (DCGAN)",
        "latent_dim": Config.DCGAN_NOISE_DIM,
        "avg_epoch_time_sec": train_out["avg_epoch_time_sec"],
        "inference_latency_ms_batch64": latency_ms,
        "history": train_out["history"],
    },
    Config.METRICS_DIR / "dcgan_results.json",
)

[DCGAN] Epoch 01/50 | D Loss: 0.6759 | G Loss: 1.6971 | Time: 143.64s
[DCGAN] Epoch 05/50 | D Loss: 0.9628 | G Loss: 1.3022 | Time: 242.87s


In [ ]:
samples = dcgan_gen.sample(16, Config.DEVICE).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")
plt.suptitle("DCGAN - 4x4 Prior Samples", fontsize=11)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 3.5))
plt.plot(train_out["history"]["g_loss"], label="Generator Loss")
plt.plot(train_out["history"]["d_loss"], label="Discriminator Loss")
plt.title("DCGAN Training Dynamics")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()